Fazendo a minha primeira chamada da LLM 

In [55]:
from langchain_google_genai import ChatGoogleGenerativeAI

from dotenv import load_dotenv 
import os

load_dotenv()

True

Criando a primeira chamada de LLM

In [56]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=os.getenv("GOOGLE_API_KEY"))  

In [57]:
llm.invoke("Quem foi Maria Antonieta?").content

'**Maria Antonieta (nascida Maria Antonia Josepha Johanna von Habsburg-Lothringen)** foi uma arquiduquesa austríaca que se tornou Rainha da França e Navarra como esposa do Rei Luís XVI. Ela é uma das figuras mais emblemáticas da história francesa, conhecida por seu estilo de vida luxuoso, sua impopularidade e seu trágico fim durante a Revolução Francesa.\n\nAqui estão os pontos chave sobre sua vida:\n\n1.  **Origem e Casamento Político:**\n    *   Nascida em 2 de novembro de 1755 em Viena, Áustria, era a décima quinta e penúltima filha da imperatriz Maria Teresa da Áustria e do imperador Francisco I do Sacro Império Romano-Germânico.\n    *   Seu casamento, aos 14 anos, em 1770, com o Delfim Luís Augusto (futuro Luís XVI) foi uma aliança política destinada a fortalecer os laços entre as casas reais da Áustria e da França, que haviam sido inimigas por séculos.\n\n2.  **Vida como Delfina e Rainha:**\n    *   Como Delfina e, a partir de 1774, como Rainha da França, Maria Antonieta rapidam

Construindo a primeira estrutura de RAG

In [58]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd
df =  pd.read_excel(DOCS_DIR / "Tickets.xlsx")
from langchain_core.documents import Document
BASE_DIR = Path().resolve()
DOCS_DIR = BASE_DIR / "docs"

Escolhendo a estrutura de embedding

In [ ]:
import os
os.environ["TRANSFORMERS_NO_LAZY_IMPORT"] = "1"
import transformers.models.bert
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [61]:

def load_pdf_vectorstore(filepath: str, save_path: str):
    loader = PyPDFLoader(DOCS_DIR / filepath)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=500)
    documents = text_splitter.split_documents(documents)
    vectorstore = FAISS.from_documents(documents, embedding)    
    vectorstore.save_local(f'vectostores/{save_path}')
    retriver = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 7}) 
    return retriver

In [62]:
retriver_perguntas_frequentes = load_pdf_vectorstore("Perguntas Frequentes.pdf", "vectorstore_perguntas_frequentes")
retriver_manual_tecnico = load_pdf_vectorstore("Manual Tecnico de Produtos.pdf", "vectorstore_manual_tecnico_produtos")
retriver_politicas_procedimentos = load_pdf_vectorstore("Politicas e Procedimentos.pdf", "vectorstore_politica_procedimentos")   

In [64]:
def load_excel_vectorstore(filepath: str, save_path: str):
    df =  pd.read_excel(DOCS_DIR / filepath)
    documents = []
    for idx, row in df.iterrows():
        text  = ' '.join([str(cell) for cell in row if pd.notna(cell)])
        documents.append(Document(page_content=text, metadata={'row': idx}))
      
    vectorstore_tickets = FAISS.from_documents(documents, embedding)    
    vectorstore_tickets.save_local('vectostores/vectorstore_tickets')
    retriver_tickets = vectorstore_tickets.as_retriever(search_type="similarity", search_kwargs={"k": 7})
    
    return retriver_tickets

In [65]:
retriver_tickets = load_excel_vectorstore("Tickets.xlsx", "vectorstore_tickets")

In [66]:
retriver_tickets.invoke('TCK-001')

[Document(id='651d8c89-6aab-4190-b091-dc816f374537', metadata={'row': 29}, page_content='TCK-030 2025-09-04 00:00:00 Construtora Beta Compressor Z150 Cliente solicita treinamento extra para técnicos Baixa (P4) Aberto Suporte Thiago'),
 Document(id='53529b03-cb05-4f5c-88a3-0d56945edd1b', metadata={'row': 1}, page_content='TCK-002 2025-08-02 00:00:00 Metalúrgica Beta Compressor Z150 Pressão abaixo do esperado durante turnos de pico Alta (P2) Aguardando peças Tec. Juliana'),
 Document(id='a37c3557-a158-47a7-81d7-8e11712f086b', metadata={'row': 27}, page_content='TCK-028 2025-09-02 00:00:00 Textil Lux Compressor Y300 Consumo elevado de energia relatado pelo cliente Média (P3) Aguardando retorno cliente Tec. Mariana'),
 Document(id='9f9f545e-8542-4500-ac8c-e81419bb5f04', metadata={'row': 32}, page_content='TCK-033 2025-09-07 00:00:00 Plásticos Delta Compressor Z150 Excesso de condensado no reservatório Média (P3) Aberto Tec. Sofia'),
 Document(id='039198b8-4128-4e22-8831-5fabde6372d7', meta